# 《사석(死石)》 1화 〈신의 한 수〉 — Wan 2.2 무료 Colab

이 노트북은 **ComfyUI + Wan 2.2 TI2V-5B**를 로컬 Colab GPU에서 실행합니다. 유료 영상 API를 호출하지 않습니다.

먼저 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 아래 셀 하나를 실행하세요. 생성된 클립은 즉시 Google Drive의 `MyDrive/SASEOK_EP01`에 저장되므로 Colab이 끊겨도 다시 실행하면 이어서 진행합니다.

최종 목표: 15장면 / 120개의 실제 AI 모션 컷 / 16:9 / 정확한 19×19 바둑판 / 최종 20분 MP4.


**기본 2컷씩 GPU 호출**: 컷이 끝날 때마다 Google Drive에 검증 후 저장합니다. 실행 상태는 `MyDrive/SASEOK_EP01/render_state/`에 기록하고, 다음 실행에서는 정상 컷을 건너뜁니다. 무료 Colab 자체의 자동 재시작은 지원되지 않습니다.


In [ ]:
import os, pathlib, subprocess, sys
from google.colab import drive

os.environ.setdefault('SASEOK_MAX_SHOTS', '2')
os.environ.setdefault('SASEOK_MAX_BATCH_MINUTES', '55')
os.environ.setdefault('SASEOK_SHOT_TIMEOUT_MINUTES', '45')

# 먼저 Drive를 마운트해 전체 실행 로그를 영구 저장합니다.
if not pathlib.Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
log_dir = pathlib.Path('/content/drive/MyDrive/SASEOK_EP01')
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / 'saseok_run.log'

repo = pathlib.Path('/content/imgs')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/chl4890620123-collab/imgs.git',str(repo)], check=True)
else:
    subprocess.run(['git','fetch','origin','main'], cwd=repo, check=True)
    subprocess.run(['git','reset','--hard','origin/main'], cwd=repo, check=True)

os.chdir(repo)
print('Python:', sys.version, flush=True)
print('Repo:', repo, flush=True)
print('Log:', log_path, flush=True)
print('GPU check:', flush=True)
try:
    gpu = subprocess.run(['nvidia-smi'], text=True, capture_output=True, check=False)
    gpu_info = gpu.stdout or gpu.stderr
except FileNotFoundError:
    gpu_info = 'No GPU detected (CPU assembly only)'
print(gpu_info, flush=True)

# stderr/stdout를 동시에 화면과 Drive 로그에 기록합니다.
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
with log_path.open('a', encoding='utf-8', buffering=1) as log:
    log.write('\n===== NEW GPU BURST / RESUME =====\n')
    log.write(f'Python: {sys.version}\n')
    log.write(f'GPU:\n{gpu_info}\n')
    proc = subprocess.Popen(
        [sys.executable, '-u', 'colab/saseok_colab_runner.py'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
        log.write(line)
    code = proc.wait()

print(f'\nSASEOK renderer exit code: {code}', flush=True)
print(f'전체 로그 저장 위치: {log_path}', flush=True)
if code != 0:
    print('Drive의 saseok_run.log 및 render_state/에서 마지막 실패 상태를 확인하세요.', flush=True)


## 재개 방법

무료 Colab 세션이 끊기거나 GPU 시간이 끝나면 나중에 다시 이 노트북을 열어 **같은 셀을 다시 실행**하면 됩니다. Drive에 있는 **검증된 컷은 건너뛰고** 다음 컷부터 생성합니다. 한 번에 기본 2컷씩 처리하고 매 컷 저장합니다. `MyDrive/SASEOK_EP01/render_state/`에서 중단된 작업을 확인할 수 있습니다.

모든 컷이 완료되면 `MyDrive/SASEOK_EP01/SASEOK_EP01_WAN22_20min.mp4`가 자동 생성됩니다.
